![Week 1 Day 2](../../assets/banners/w1d2.png)

# Week 1 Day 2 - Solution: Batch on a Budget

The Day 2 exercise: pick three questions from Asha's question paper, **estimate** the cost before calling anything, **batch** them in one go, then compare your estimate with what the provider actually billed.

> 💡 Try the exercise yourself first (it's at the end of `week1/day2.ipynb`). Struggling a little is where the learning happens! Then come back and compare.

In [ ]:
# Imports and environment first, all in one place

import os

import tiktoken
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

from gradewise.config import settings

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: Three questions and an estimate (offline)

The prices are placeholders: replace them with your model's real ones from your provider's pricing page. For the output, we estimate: we ask for about 60 words, which is roughly 80 tokens.

In [ ]:
PRICE_PER_MILLION = {"input": 0.75, "output": 4.50}  # ✋ replace with your model's real prices (USD)
EXPECTED_OUTPUT_TOKENS = 80  # 💡 we ask for ~60 words; 1 token ≈ ¾ of a word

questions = [
    "What is a system call? Explain why the CPU switches from user mode to kernel mode when a system call is made.",
    "Describe the TCP three-way handshake used to open a connection.",
    "State and briefly explain the four ACID properties of a database transaction.",
]
prompts = [f"Write a model answer of about 60 words for this exam question:\n{q}" for q in questions]

encoding = tiktoken.get_encoding("o200k_base")
estimated_in = [len(encoding.encode(p)) for p in prompts]


def cost(tokens_in: int, tokens_out: int) -> float:
    return (tokens_in * PRICE_PER_MILLION["input"] + tokens_out * PRICE_PER_MILLION["output"]) / 1_000_000


estimate = sum(cost(n, EXPECTED_OUTPUT_TOKENS) for n in estimated_in)
print("Estimated input tokens per prompt:", estimated_in)
print(f"Estimated cost for all three: ${estimate:.6f}")

## Part 2: Batch them

In [ ]:
llm = init_chat_model(settings.chat_model)
replies = llm.batch(prompts, config={"max_concurrency": 3})

for q, r in zip(questions, replies, strict=True):
    print(f"❓ {q}\n💬 {r.content}\n")

## Part 3: Estimate vs reality

In [ ]:
actual = 0.0
print(f"{'':<4}{'in (est)':>9}{'in (real)':>10}{'out (est)':>10}{'out (real)':>11}")
for i, (n, r) in enumerate(zip(estimated_in, replies, strict=True), start=1):
    u = r.usage_metadata
    actual += cost(u["input_tokens"], u["output_tokens"])
    print(f"Q{i:<3}{n:>9}{u['input_tokens']:>10}{EXPECTED_OUTPUT_TOKENS:>10}{u['output_tokens']:>11}")

print(f"\nEstimated ${estimate:.6f} · actual ${actual:.6f} · ratio {actual / estimate:.1f}×")

**What you'll probably see:** the input estimate is within a few tokens (the provider adds a little message formatting). The *output* is where estimates go wrong: models rarely stop at exactly 60 words, and **reasoning models also bill their hidden "thinking" tokens as output**. Look for `output_token_details` → `reasoning` in `usage_metadata`. That's why GradeWise logs real usage from every call rather than trusting estimates.

In [ ]:
print(replies[0].usage_metadata)  # 💡 look for output_token_details → reasoning

## Bonus: stream the longest answer

In [ ]:
longest = max(range(len(replies)), key=lambda i: replies[i].usage_metadata["output_tokens"])
for chunk in llm.stream(prompts[longest]):
    print(chunk.content, end="", flush=True)